# Slice viewer for one NIfTI file (3D Slicer style)

Pick any `.nii` / `.nii.gz` file under `data/` or `runs/` and click **Load**. You get three views side by side, each with its own slider:

| View | Cuts along | Screen left / right | Screen top / bottom | Colour |
|---|---|---|---|---|
| **Axial** | S-I (z) | R / L | A / P | red |
| **Coronal** | A-P (y) | R / L | S / I | green |
| **Sagittal** | R-L (x) | A / P | S / I | yellow |

The axial and coronal views follow the radiological convention, as 3D Slicer does: the patient's right is on the left of the screen. Each view's edge letters show the direction.

- **Sliders**: drag, or click a slider and use the arrow keys, to step one slice at a time. The coloured lines in the other two views show where the current slices are (**Crosshair**).
- **Mask** (optional): a label or prediction file, drawn in colour on top of the MRI. Picking an image from an `imagesTr/` folder fills in the matching label automatically. The mask is matched to the image by position in mm, so crops and padded predictions line up.
  - anterior/posterior (MSD, predictions): orange = anterior, blue = posterior
  - subfields (external dataset): gold = CA1-3, green = subiculum, purple = CA4-DG (names inferred from the atlas)
- **Contrast %**: the darkest and brightest grey, as percentiles of the tissue intensities (like window/level).
- **Opacity**: how strongly the mask covers the MRI (0 hides it).
- **Size**: the largest panel's size in pixels. Pixels are square in mm, so the views have true proportions.

The line under the views gives the crosshair position in mm, the intensity there and the label. To compare the predictions of several runs, use `compare_slices.ipynb`. Run all cells, then use the controls under the last cell. Research use only, not for diagnosis.

In [ ]:
# Work from the project root (so data/ and runs/ resolve) and import hippo from src/
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "hippo").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

import ipywidgets as w
import numpy as np
from IPython.display import display

from hippo.slices import (ERRORS, NONE, VIEWS, Scan, error_map, guess_mask, label_colours, legend_html,
                          list_nifti, mm_per_px, on_grid, rgb_colours)

print(f"project: {ROOT}")

In [ ]:
files = list_nifti()
wide = w.Layout(width="95%")
search = w.Text(placeholder="type to filter, e.g.  hippocampus_135   or   s03 t1w_standard", description="Filter", layout=wide)
image_dd = w.Dropdown(options=files, description="Image", layout=wide)
mask_dd = w.Dropdown(options=[NONE] + files, description="Mask", layout=wide)
load_btn = w.Button(description="Load", button_style="primary", icon="eye")
contrast = w.IntRangeSlider(value=(1, 99), min=0, max=100, description="Contrast %")
opacity = w.FloatSlider(value=0.5, min=0, max=1, step=0.05, description="Opacity")
crosshair = w.Checkbox(value=True, description="Crosshair", indent=False)
size = w.IntSlider(value=340, min=160, max=640, step=20, description="Size px")
legend = w.HTML()
info = w.HTML()
status = w.HTML()

pictures, sliders, positions, panels = [], [], [], []
for name, axis, _, colour in VIEWS:
    pictures.append(w.Image(format="png"))
    sliders.append(w.IntSlider(description=name, continuous_update=True, disabled=True, layout=w.Layout(width="98%")))
    positions.append(w.HTML())
    panels.append(w.VBox([w.HTML(f"<b style='color:{colour}'>{name}</b>"), pictures[-1], sliders[-1], positions[-1]],
                         layout=w.Layout(border=f"2px solid {colour}", padding="4px", margin="2px",
                                         align_items="center")))
state = {"scan": None, "pos": None, "ready": False}


def render(_=None):
    scan, pos = state["scan"], state["pos"]
    if not state["ready"]:
        return
    scale = mm_per_px(scan.data.shape, scan.spacing, size.value)
    for i, view in enumerate(VIEWS):
        pictures[i].value = scan.render(view, pos, contrast.value, opacity.value, crosshair.value, scale)
        axis = view[1]
        positions[i].value = (f"<small>slice {pos[axis] + 1} / {scan.data.shape[axis]}"
                              f" &nbsp; {'xyz'[axis]} = {scan.mm(axis, pos[axis]):.1f} mm</small>")
    info.value = f"<code>{scan.describe(pos)}</code>"


def on_slide(i, change):
    if state["ready"]:
        state["pos"][VIEWS[i][1]] = change["new"]
        render()


def on_load(_=None):
    if not image_dd.value:
        status.value = "No file matches the filter."
        return
    status.value = f"Loading {image_dd.value} ..."
    state["ready"] = False
    try:
        scan = Scan(image_dd.value, mask_dd.value)
    except Exception as e:  # show the problem under the controls instead of a long traceback
        status.value = f"<span style='color:#e8605a'>Could not load this file: {e}</span>"
        return
    state.update(scan=scan, pos=list(scan.start))
    for (_, axis, _, _), slider in zip(VIEWS, sliders):
        slider.max = scan.data.shape[axis] - 1
        slider.value = scan.start[axis]
        slider.disabled = False
    legend.value = legend_html(scan.colours)
    shape, spacing = scan.data.shape, scan.spacing
    status.value = (f"{Path(image_dd.value).name}: {shape[0]} x {shape[1]} x {shape[2]} voxels, "
                    f"{spacing[0]:.2g} x {spacing[1]:.2g} x {spacing[2]:.2g} mm")
    state["ready"] = True
    render()


def on_filter(change):
    terms = change["new"].lower().split()
    match = [f for f in files if all(t in f.lower() for t in terms)]
    image_dd.options = match
    mask_dd.options = [NONE] + match
    on_image({"new": image_dd.value})


def on_image(change):
    if not change["new"]:
        return
    guess = guess_mask(change["new"])
    if guess not in mask_dd.options:
        mask_dd.options = list(mask_dd.options) + [guess]
    mask_dd.value = guess


search.observe(on_filter, names="value")
image_dd.observe(on_image, names="value")
load_btn.on_click(on_load)
for control in (contrast, opacity, crosshair, size):
    control.observe(render, names="value")
for i, slider in enumerate(sliders):
    slider.observe(lambda change, i=i: on_slide(i, change), names="value")

on_image({"new": image_dd.value})
display(w.VBox([search, image_dd, mask_dd, w.HBox([load_btn, status]),
                w.HBox([contrast, opacity, crosshair]), w.HBox([size, legend]),
                w.HBox(panels, layout=w.Layout(flex_flow="row wrap")), info]))